# Skin Lesion Classification with DINOv2

Train a DINOv2 vision foundation model on the ISIC 2019 skin lesion dataset.

**Runtime:** Go to `Runtime → Change runtime type → T4 GPU`

This notebook clones the repo and calls into the project scripts — all core logic lives in `src/`.

## 1. Setup

In [ ]:
# Verify GPU is available
!nvidia-smi

In [ ]:
# Clone the repo (only needs to run once per session)
import os

REPO_DIR = "/content/isic-skin-lesion-classification"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/mtang4/isic-skin-lesion-classification.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

## 2. Download and Prepare Data

Downloads the ISIC 2019 dataset (~9GB) and creates stratified train/val/test splits.

This takes ~10-15 minutes on Colab. The data is stored in `/content/isic-skin-lesion-classification/data/` and persists for the session.

In [ ]:
!python scripts/download_data.py --data_dir ./data

In [ ]:
# Quick sanity check: view class distribution
import pandas as pd

weights = pd.read_csv("data/class_weights.csv")
print(weights.to_string(index=False))
print(f"\nTotal training samples: {weights['count'].sum()}")

In [ ]:
# Preview some training images
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import random

CLASS_NAMES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, cls in zip(axes.flat, CLASS_NAMES):
    cls_dir = Path(f"data/train/{cls}")
    img_path = random.choice(list(cls_dir.glob("*.jpg")))
    ax.imshow(Image.open(img_path))
    ax.set_title(cls, fontsize=14)
    ax.axis("off")
plt.suptitle("Sample Training Images (one per class)", fontsize=16)
plt.tight_layout()
plt.show()

## 3. Train

### Option A: Linear Probe (recommended first)

Freezes the DINOv2 backbone and only trains the classification head.  
~10 min on a T4 for 30 epochs.

In [ ]:
!python src/train.py \
    --config configs/default.yaml \
    --mode linear_probe \
    --run_name linear_probe_base

### Option B: Fine-Tune

Unfreezes the backbone with a low learning rate. Better accuracy, ~1hr on a T4.

In [ ]:
# Uncomment to run fine-tuning
# !python src/train.py \
#     --config configs/default.yaml \
#     --mode fine_tune \
#     --epochs 15 \
#     --lr 1e-4 \
#     --batch_size 32 \
#     --run_name finetune_base

## 4. Evaluate

In [ ]:
# Update the checkpoint path to match your run name
CHECKPOINT = "checkpoints/linear_probe_base/best_model.pt"

!python src/evaluate.py --checkpoint {CHECKPOINT} --data_dir ./data

In [ ]:
# Display the generated plots
from IPython.display import display, Image as IPImage
from pathlib import Path

ckpt_dir = Path(CHECKPOINT).parent

for plot in ["confusion_matrix.png", "per_class_metrics.png"]:
    plot_path = ckpt_dir / plot
    if plot_path.exists():
        print(f"\n{plot}")
        display(IPImage(filename=str(plot_path), width=800))

## 5. TensorBoard (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/

## 6. Save Checkpoint to Google Drive (optional)

Colab sessions are ephemeral. Save your best checkpoint to Drive so you don't lose it.

In [ ]:
# Uncomment to mount Drive and copy checkpoint
# from google.colab import drive
# drive.mount("/content/drive")
#
# import shutil
# drive_dest = "/content/drive/MyDrive/ml-checkpoints/isic-dinov2/"
# os.makedirs(drive_dest, exist_ok=True)
# shutil.copy2(CHECKPOINT, drive_dest)
# print(f"Saved to {drive_dest}")